In [72]:
# https://www.kaggle.com/competitions/titanic/data                          TITANIC
# https://www.kaggle.com/datasets/adilshamim8/student-depression-dataset    DEPRESSÃO
# https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set          CARRO

!pip install seaborn

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def preprocess_data(df, target_col):
    # Limpeza de valores nulos
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

    # Codificação de variáveis categóricas
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    # Features e alvo
    X = df.drop(columns=target_col)
    y = df[target_col]

    # Padronização
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Tensores PyTorch
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.long)

    return X_tensor, y_tensor

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, 10)  # número máximo de classes
        )

    def forward(self, x):
        return self.model(x)

def train_and_evaluate(model, train_loader, test_loader, output_dim, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)[:, :output_dim]
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

    # Avaliação
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = model(X_batch)[:, :output_dim]
            _, predicted = torch.max(output, 1)
            y_true.extend(y_batch.numpy())
            y_pred.extend(predicted.numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Acurácia: {acc:.4f} | Precisão: {prec:.4f} | Recall: {rec:.4f} | F1-score: {f1:.4f}")
    return acc, prec, rec, f1

def cross_validate(X_tensor, y_tensor, k=5, hidden_dim=64):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    accs, precs, recs, f1s = [], [], [], []
    # Número de classes diferentes no alvo
    output_dim = len(torch.unique(y_tensor))

    # Loop por cada fold da validação cruzada
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_tensor)):
        print(f"\nFold {fold+1}/{k}")

        # Separa os dados de treino e teste com base nos índices do fold
        X_train, y_train = X_tensor[train_idx], y_tensor[train_idx]
        X_test, y_test = X_tensor[test_idx], y_tensor[test_idx]

        # Cria datasets e dataloaders para treino e teste
        train_ds = TensorDataset(X_train, y_train)
        test_ds = TensorDataset(X_test, y_test)

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

        model = MLP(X_tensor.shape[1], hidden_dim)
        # Treina e avalia o modelo nesse fold, retornando métricas
        acc, prec, rec, f1 = train_and_evaluate(model, train_loader, test_loader, output_dim)
        accs.append(acc); precs.append(prec); recs.append(rec); f1s.append(f1)

    print("\nMÉDIAS GERAIS:")
    print(f"Acurácia Média: {np.mean(accs):.4f}")
    print(f"Precisão Média: {np.mean(precs):.4f}")
    print(f"Recall Médio: {np.mean(recs):.4f}")
    print(f"F1-score Médio: {np.mean(f1s):.4f}")

In [73]:
df_titanic = pd.read_csv("titanic.csv")

colunas_remover = ["PassengerId", "Name", "Ticket", "Cabin"]
df_titanic = df_titanic.drop(columns=colunas_remover)

X_titanic, y_titanic = preprocess_data(df_titanic, target_col="Survived")

print("Titanic")
cross_validate(X_titanic, y_titanic, k=5)

Titanic

Fold 1/5
Acurácia: 0.8324 | Precisão: 0.8347 | Recall: 0.8324 | F1-score: 0.8295

Fold 2/5
Acurácia: 0.7921 | Precisão: 0.7906 | Recall: 0.7921 | F1-score: 0.7867

Fold 3/5
Acurácia: 0.8820 | Precisão: 0.8851 | Recall: 0.8820 | F1-score: 0.8799

Fold 4/5
Acurácia: 0.8034 | Precisão: 0.8099 | Recall: 0.8034 | F1-score: 0.7932

Fold 5/5
Acurácia: 0.8202 | Precisão: 0.8182 | Recall: 0.8202 | F1-score: 0.8182

MÉDIAS GERAIS:
Acurácia Média: 0.8260
Precisão Média: 0.8277
Recall Médio: 0.8260
F1-score Médio: 0.8215


In [74]:
df_dep = pd.read_csv("student_depression_dataset.csv")  # nome real do arquivo após upload

colunas_remover = ["id"]
df_dep = df_dep.drop(columns=colunas_remover)

# Supondo que a última coluna seja o alvo
target_dep = df_dep.columns[-1]
X_dep, y_dep = preprocess_data(df_dep, target_col=target_dep)

print("Depressão Estudantil")
cross_validate(X_dep, y_dep, k=5)

Depressão Estudantil

Fold 1/5
Acurácia: 0.8352 | Precisão: 0.8347 | Recall: 0.8352 | F1-score: 0.8348

Fold 2/5
Acurácia: 0.8523 | Precisão: 0.8519 | Recall: 0.8523 | F1-score: 0.8515

Fold 3/5
Acurácia: 0.8507 | Precisão: 0.8504 | Recall: 0.8507 | F1-score: 0.8499

Fold 4/5
Acurácia: 0.8403 | Precisão: 0.8398 | Recall: 0.8403 | F1-score: 0.8391

Fold 5/5
Acurácia: 0.8534 | Precisão: 0.8530 | Recall: 0.8534 | F1-score: 0.8529

MÉDIAS GERAIS:
Acurácia Média: 0.8464
Precisão Média: 0.8460
Recall Médio: 0.8464
F1-score Médio: 0.8456


In [76]:
df_car = pd.read_csv("car_evaluation.csv")

# última coluna seja o alvo
target_car = df_car.columns[-1]
X_car, y_car = preprocess_data(df_car, target_col=target_car)

print("Evolução Carros")
cross_validate(X_car, y_car, k=5)

Evolução Carros

Fold 1/5
Acurácia: 0.9306 | Precisão: 0.9376 | Recall: 0.9306 | F1-score: 0.9144

Fold 2/5
Acurácia: 0.9277 | Precisão: 0.9283 | Recall: 0.9277 | F1-score: 0.9183

Fold 3/5
Acurácia: 0.9362 | Precisão: 0.9460 | Recall: 0.9362 | F1-score: 0.9215

Fold 4/5
Acurácia: 0.9333 | Precisão: 0.9305 | Recall: 0.9333 | F1-score: 0.9238

Fold 5/5
Acurácia: 0.9420 | Precisão: 0.9460 | Recall: 0.9420 | F1-score: 0.9380

MÉDIAS GERAIS:
Acurácia Média: 0.9340
Precisão Média: 0.9377
Recall Médio: 0.9340
F1-score Médio: 0.9232
